# 01 - EDA Preliminar RAW

**Proyecto**: Research_AI_law  
**Fase**: 2 - Entendimiento de datos / EDA preliminar RAW  
**Fecha**: 2026-05-05

## Objetivo

Auditoria exploratoria neutral, reproducible y trazable de las 8 fuentes unificadas disponibles, con foco en cobertura, calidad, estructura, granularidad e integrabilidad.

**Sin definir todavia**: variables dependientes, independientes, tratamientos, controles o features.

## Principios rectores

1. **Datos reales, cero sinteticos** - cada celda auditable por un humano
2. **Neutralidad analitica** - no se clasifica aun como X/Y/feature
3. **Trazabilidad total** - saber de que archivo, hoja y variable proviene cada dato
4. **No perdida silenciosa** - nada se elimina sin quedar registrado
5. **Reproducibilidad** - todos los conteos regenerables desde este notebook

## Estructura del notebook (13 bloques)

```text
0. Setup y funciones utilitarias
1. Etapa 0 - Preparacion del entorno
2. Etapa 1 - Inventario de fuentes
3. Etapa 2 - Inventario de tablas
4. Etapa 3 - Auditoria de esquema
5. Etapa 4 - Auditoria geografica
6. Etapa 5 - Auditoria temporal
7. Etapa 6 - Completitud
8. Etapa 7 - Perfil estadistico univariado
9. Etapa 8 - Diagnostico de calidad
10. Etapa 9 - Matriz preliminar pais x fuente
11. Etapa 10 - Matriz preliminar pais x atributo
12. Etapa 11 - Recomendaciones para wrangling
13. Etapa 12 - Resumen ejecutivo
```

## Fuentes auditadas (8)

| # | Fuente | Archivo | Formato |
|---|---|---|---|
| 1 | IAPP | `IAPP/iapp_dataset_unificado.xlsx` | XLSX |
| 2 | Oxford | `Oxford Insights/data_index/UNIFICADO/Oxford_Insights_Unificado.xlsx` | XLSX |
| 3 | World Bank | `World Bank WDI/WB_WDI_WGI_unificado.xlsx` | XLSX |
| 4 | WIPO | `WIPO Global Innovation Index/data_wipo_gii/WIPO_GII_2021_2025_UNIFICADO.xlsx` | XLSX |
| 5 | Microsoft | `MICROSOFT/Microsoft_AI_Reports_Data_unificado.xlsx` | XLSX |
| 6 | OECD | `OECD/OECD_Data_Central_unificado.xlsx` | XLSX |
| 7 | Stanford | `STANFORD AI INDEX 26/.../stanford_ai_index_2026_unificado.csv` | CSV |
| 8 | Anthropic | `ANTROPHIC/data_unificada/antrophic_economic_index.xlsx` | XLSX |


In [55]:
# =============================================================================
# BLOQUE 0 - SETUP: Imports, constantes y funciones utilitarias
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import json
from datetime import datetime

# Visualizaciones
import matplotlib.pyplot as plt
import seaborn as sns

# Configuracion de matplotlib
plt.style.use('default')
sns.set_palette("husl")
# matplotlib inline removed for non-interactive execution

# =============================================================================
# CONSTANTES
# =============================================================================

ROOT = Path("/home/pablo/Research_LeyIA_DataScience/Research_AI_law")
OUTPUT_DIR = ROOT / "outputs" / "eda_preliminar"

# Asegurar que el directorio de outputs existe
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Definicion de las 8 fuentes con sus archivos canonicos
SOURCES = {
    "iapp": {
        "name": "IAPP",
        "organization": "International Association of Privacy Professionals",
        "file": ROOT / "IAPP" / "iapp_dataset_unificado.xlsx",
        "format": "XLSX",
        "primary_table": "datos",
        "obs_unit": "Pais",
        "geo_id": "iso3",
        "time_expected": "Cross-section (snapshot 2026-01)",
    },
    "microsoft": {
        "name": "Microsoft",
        "organization": "Microsoft",
        "file": ROOT / "MICROSOFT" / "Microsoft_AI_Reports_Data_unificado.xlsx",
        "format": "XLSX",
        "primary_table": "1_AI_Diffusion",
        "obs_unit": "Pais / region / estadistica",
        "geo_id": "Economy (country name, no ISO3)",
        "time_expected": "Snapshot H1/H2 2025 + estadisticas puntuales",
    },
    "oxford": {
        "name": "Oxford Insights",
        "organization": "Oxford Insights",
        "file": ROOT / "Oxford Insights" / "data_index" / "UNIFICADO" / "Oxford_Insights_Unificado.xlsx",
        "format": "XLSX",
        "primary_table": "Consolidado",
        "obs_unit": "Pais-ano",
        "geo_id": "iso3",
        "time_expected": "Panel 2019-2025",
    },
    "wb": {
        "name": "World Bank WDI/WGI",
        "organization": "World Bank",
        "file": ROOT / "World Bank WDI" / "WB_WDI_WGI_unificado.xlsx",
        "format": "XLSX",
        "primary_table": "MATRIZ_COMPLETA",
        "obs_unit": "Pais",
        "geo_id": "iso3",
        "time_expected": "Panel ancho 2018-2025",
    },
    "wipo": {
        "name": "WIPO GII",
        "organization": "World Intellectual Property Organization",
        "file": ROOT / "WIPO Global Innovation Index" / "data_wipo_gii" / "WIPO_GII_2021_2025_UNIFICADO.xlsx",
        "format": "XLSX",
        "primary_table": "MATRIZ_GII",
        "obs_unit": "Pais / indicador-ano",
        "geo_id": "ISO3",
        "time_expected": "Panel ancho 2021-2025",
    },
    "stanford": {
        "name": "Stanford AI Index",
        "organization": "Stanford HAI",
        "file": ROOT / "STANFORD AI INDEX 26" / "PUBLIC DATA_ 2026 AI INDEX REPORT" / "completo" / "stanford_ai_index_2026_unificado.csv",
        "format": "CSV",
        "primary_table": "stanford_ai_index_2026_unificado",
        "obs_unit": "Entidad-figura-variable",
        "geo_id": "iso3",
        "time_expected": "Panel long por figura/variable/entidad/ano",
    },
    "oecd": {
        "name": "OECD",
        "organization": "OECD",
        "file": ROOT / "OECD" / "OECD_Data_Central_unificado.xlsx",
        "format": "XLSX",
        "primary_table": "Multiple (9 data sheets)",
        "obs_unit": "Pais-ano",
        "geo_id": "iso3",
        "time_expected": "Panel pais-ano, multiples series",
    },
    "anthropic": {
        "name": "Anthropic",
        "organization": "Anthropic",
        "file": ROOT / "ANTROPHIC" / "data_unificada" / "antrophic_economic_index.xlsx",
        "format": "XLSX",
        "primary_table": "aei_metrics_wide",
        "obs_unit": "Geografia-fecha-plataforma",
        "geo_id": "geo_id (ISO2) + dim_geography crosswalk",
        "time_expected": "Ventanas temporales por geografia/plataforma",
    },
}

# Prefijos para naming de variables
PREFIXES = {
    "iapp": "iapp_",
    "microsoft": "ms_",
    "oxford": "oxford_",
    "wb": "wb_",
    "wipo": "wipo_",
    "stanford": "stanford_",
    "oecd": "oecd_",
    "anthropic": "anthropic_",
}

print(f"ROOT: {ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Fuentes configuradas: {len(SOURCES)}")
print("OK - Setup completo")


ROOT: /home/pablo/Research_LeyIA_DataScience/Research_AI_law
OUTPUT_DIR: /home/pablo/Research_LeyIA_DataScience/Research_AI_law/outputs/eda_preliminar
Fuentes configuradas: 8
OK - Setup completo


In [56]:
# =============================================================================
# FUNCIONES UTILITARIAS
# =============================================================================

def read_table_preview(path, sheet_name=None, nrows=5):
    result = {"path": str(path), "sheet": sheet_name, "exists": __import__('os').path.exists(path)}
    if not result["exists"]:
        return result
    try:
        if str(path).endswith('.csv'):
            df_sample = __import__('pandas').read_csv(path, nrows=nrows)
            with open(path, 'r', encoding='utf-8', errors='replace') as f:
                n_rows = sum(1 for _ in f) - 1
            df_full = __import__('pandas').read_csv(path, nrows=1000)
        else:
            if sheet_name:
                df_sample = __import__('pandas').read_excel(path, sheet_name=sheet_name, nrows=nrows, engine='openpyxl')
                df_full = __import__('pandas').read_excel(path, sheet_name=sheet_name, engine='openpyxl')
                n_rows = len(df_full)
            else:
                xl = __import__('pandas').ExcelFile(path, engine='openpyxl')
                first_sheet = xl.sheet_names[0]
                df_sample = __import__('pandas').read_excel(path, sheet_name=first_sheet, nrows=nrows, engine='openpyxl')
                df_full = __import__('pandas').read_excel(path, sheet_name=first_sheet, engine='openpyxl')
                n_rows = len(df_full)
                result["sheet_used"] = first_sheet
        result["shape"] = (n_rows, len(df_full.columns))
        result["columns"] = list(df_full.columns)
        result["dtypes"] = {str(c): str(dt) for c, dt in df_full.dtypes.items()}
        result["non_null_counts"] = {str(c): int(df_full[c].notna().sum()) for c in df_full.columns}
    except Exception as e:
        result["error"] = str(e)
    return result

def infer_column_type(series):
    col_name = str(series.name).lower()
    n_non_null = series.notna().sum()
    n_unique = series.nunique(dropna=True)
    if n_non_null == 0: return "unknown"
    geo_cols = ['iso3', 'iso', 'country_code', 'economy_code', 'geo_id', 'countryiso3code']
    if col_name in geo_cols: return "geo_identifier"
    if any(x in col_name for x in ['iso3', 'country_code', 'economy', 'pais', 'country', 'nation']):
        if n_unique <= 250 and n_non_null > 0: return "geo_identifier"
    if col_name in ['year', 'date', 'fecha', 'period', 'date_start', 'date_end', 'ano', 'anio']: return "time"
    if __import__('re').fullmatch(r'(19|20)\d{2}', col_name): return "time"
    if col_name in ['id', 'index', 'num', 'fig_id', 'variable_id']: return "identifier"
    non_null = series.dropna()
    if non_null.astype(str).str.startswith('http').any(): return "url"
    if 'date' in col_name or 'fecha' in col_name:
        try:
            __import__('pandas').to_datetime(non_null, errors='raise')
            return "date"
        except: pass
    if n_unique == 2:
        unique_vals = set(non_null.unique())
        if unique_vals.issubset({0, 1, True, False, '0', '1', 'yes', 'no', 'si', 'true', 'false'}):
            return "binary"
    try:
        numeric = __import__('pandas').to_numeric(non_null, errors='coerce')
        if numeric.notna().sum() / n_non_null > 0.9:
            if n_unique <= 10 and n_unique > 2: return "ordinal"
            if all(numeric.dropna() == numeric.dropna().astype(int)): return "numeric_integer"
            return "numeric_continuous"
    except: pass
    if non_null.astype(str).str.len().mean() > 100: return "text_long"
    if n_unique <= min(50, n_non_null * 0.5): return "categorical"
    if any(x in col_name for x in ['source', 'fuente', 'audit', 'sha256', 'metadata', 'note', 'nota']): return "metadata"
    return "unknown"

def detect_geo_columns(columns):
    """Devuelve columnas geograficas priorizando codigos canonicos sobre nombres/descriptores."""
    exact_priority = ['iso3', 'countryiso3code', 'country_code', 'economy_code', 'geo_id', 'iso2', 'iso']
    name_priority = ['pais', 'country', 'economy', 'jurisdiction', 'entity_name_std', 'entity_name_raw', 'entity']
    region_priority = ['region', 'nation', 'geo']
    scored = []
    for c in columns:
        c_str = str(c)
        c_lower = c_str.lower().strip()
        score = None
        if c_lower in exact_priority:
            score = exact_priority.index(c_lower)
        elif any(k == c_lower or k in c_lower for k in name_priority):
            score = 100 + next(i for i, k in enumerate(name_priority) if k == c_lower or k in c_lower)
        elif any(k == c_lower or k in c_lower for k in region_priority):
            score = 200 + next(i for i, k in enumerate(region_priority) if k == c_lower or k in c_lower)
        if score is not None:
            scored.append((score, c_str))
    return [c for _, c in sorted(scored, key=lambda x: x[0])]

def detect_time_columns(columns):
    """Detecta columnas que son tiempo, no medidas cuyo nombre contiene un ano."""
    exact_time = {'year', 'anio', 'ano', 'date', 'fecha', 'period', 'time', 'date_start', 'date_end'}
    candidates = []
    for c in columns:
        c_str = str(c)
        c_lower = c_str.lower().strip()
        if c_lower in exact_time or c_lower.endswith('_year') or c_lower.endswith('_date'):
            candidates.append(c_str)
    return candidates

def classify_entity(raw_code, raw_name):
    code_str = str(raw_code).upper().strip() if __import__('pandas').notna(raw_code) else ""
    name_str = str(raw_name).lower().strip() if __import__('pandas').notna(raw_name) else ""
    non_countries = {'WLD', 'WOR', 'WORLD', 'GLO', 'GLOBAL', 'EU', 'EUU', 'EU27', 'EU28', 'OECD', 'OED', 'OPEC', 'ASEAN',
        'SSA', 'SSF', 'SST', 'MENA', 'MEA', 'MNA', 'LAC', 'LCN', 'EAP', 'EAS', 'ECA', 'ECS', 'SAR', 'SAS',
        'ARB', 'CEB', 'CSS', 'EMU', 'FCS', 'HPC', 'IBD', 'IBT', 'IDA', 'IDX', 'INX', 'LTE', 'LMY', 'MIC',
        'NAC', 'OSS', 'PRE', 'PSS', 'PST', 'TEA', 'TEC', 'TLA', 'TMN', 'TSA', 'TSS',
        'HIGH_INCOME', 'UPPER_MIDDLE', 'LOWER_MIDDLE', 'LOW_INCOME'}
    territory_codes = {'HKG', 'MAC', 'PRI', 'VIR', 'GUM', 'ASM', 'ABW', 'CUW', 'CYM', 'BMU', 'FRO', 'GRL',
        'GIB', 'IMN', 'JEY', 'GGY', 'TCA', 'VGB', 'AIA', 'MSR', 'NCL', 'PYF', 'GLP', 'MTQ', 'REU', 'MYT', 'GUF'}
    region_keywords = ['region', 'income', 'world', 'global', 'oecd', 'union', 'africa', 'asia', 'europe', 'america']
    subnational_keywords = ['state', 'province', 'region of', 'county', 'municipality']
    if code_str in territory_codes:
        return "territory_iso3"
    if len(code_str) == 3 and code_str.isalpha() and code_str not in non_countries:
        return "country_iso3"
    if code_str in non_countries or any(kw in name_str for kw in region_keywords):
        if code_str in ['WLD', 'WORLD', 'GLOBAL']: return "global"
        if code_str in ['EU', 'EU27', 'EU28']: return "organization_or_group"
        return "region"
    if any(kw in name_str for kw in subnational_keywords): return "subnational"
    return "unknown"

def compute_completeness(df):
    n_rows = len(df)
    col_completeness = {}
    for c in df.columns:
        n_non_null = df[c].notna().sum()
        col_completeness[str(c)] = {
            "n_non_null": int(n_non_null),
            "n_null": int(n_rows - n_non_null),
            "pct_complete": round(n_non_null / n_rows * 100, 2) if n_rows > 0 else 0
        }
    row_completeness = df.notna().sum(axis=1)
    row_stats = {
        "mean_attributes": round(row_completeness.mean(), 2),
        "median_attributes": round(row_completeness.median(), 2),
        "min_attributes": int(row_completeness.min()),
        "max_attributes": int(row_completeness.max()),
    }
    return {"n_rows": n_rows, "n_cols": len(df.columns), "by_column": col_completeness, "by_row": row_stats}

def detect_quality_issues(df, source_id, table_id):
    issues = []
    issue_counter = [0]
    def add_issue(issue_type, severity, variable, description, evidence, action, phase):
        issue_counter[0] += 1
        issues.append({
            "issue_id": f"{source_id}_{table_id}_{issue_counter[0]:04d}",
            "source_id": source_id, "table_id": table_id, "variable": variable, "entity": "",
            "issue_type": issue_type, "severity": severity, "description": description,
            "evidence": evidence, "recommended_action": action, "phase_to_resolve": phase,
        })
    n_rows = len(df)
    for c in df.columns:
        n_non_null = df[c].notna().sum()
        col_name = str(c)
        if n_non_null == 0:
            add_issue("empty_column", "alta", col_name, f"Columna '{col_name}' esta 100% vacia",
                f"0/{n_rows} valores no nulos", "Excluir de matriz madre o investigar", "wrangling")
        elif n_non_null / n_rows < 0.1:
            add_issue("mostly_missing_column", "media", col_name,
                f"Columna '{col_name}' tiene <10% de datos", f"{n_non_null}/{n_rows} ({n_non_null/n_rows*100:.1f}%)",
                "Evaluar si vale la pena conservar", "wrangling")
    for c in df.columns:
        n_unique = df[c].nunique(dropna=True)
        if n_unique == 1 and df[c].notna().sum() > 0:
            add_issue("constant_column", "baja", str(c), f"Columna '{c}' tiene un unico valor",
                f"1 valor unico en {df[c].notna().sum()} filas", "Evaluar utilidad analitica", "wrangling")
    return issues

print("OK - Funciones utilitarias cargadas")

OK - Funciones utilitarias cargadas


## BLOQUE 1 - ETAPA 0: Preparacion del entorno

Verificamos que los 8 archivos unificados existen.

In [57]:
# =============================================================================
# BLOQUE 1 - ETAPA 0: Verificar existencia de archivos
# =============================================================================

verification_results = []
for src_id, info in SOURCES.items():
    path = info["file"]
    exists = path.exists()
    size_mb = round(path.stat().st_size / (1024*1024), 2) if exists else None
    verification_results.append({
        "source_id": src_id, "source_name": info["name"],
        "file": str(path.relative_to(ROOT)), "exists": exists,
        "size_mb": size_mb, "status": "available" if exists else "missing",
    })

df_verification = pd.DataFrame(verification_results)
print("="*70)
print("VERIFICACION DE ARCHIVOS CANONICOS")
print("="*70)
print(df_verification.to_string(index=False))
print()
missing = df_verification[df_verification["exists"] == False]
if len(missing) > 0:
    print(f"ADVERTENCIA: {len(missing)} archivo(s) faltante(s)")
else:
    print("OK - Los 8 archivos canonicos estan presentes")

VERIFICACION DE ARCHIVOS CANONICOS
source_id        source_name                                                                                                 file  exists  size_mb    status
     iapp               IAPP                                                                     IAPP/iapp_dataset_unificado.xlsx    True     0.02 available
microsoft          Microsoft                                                   MICROSOFT/Microsoft_AI_Reports_Data_unificado.xlsx    True     0.03 available
   oxford    Oxford Insights                                  Oxford Insights/data_index/UNIFICADO/Oxford_Insights_Unificado.xlsx    True     0.92 available
       wb World Bank WDI/WGI                                                             World Bank WDI/WB_WDI_WGI_unificado.xlsx    True     1.04 available
     wipo           WIPO GII                         WIPO Global Innovation Index/data_wipo_gii/WIPO_GII_2021_2025_UNIFICADO.xlsx    True     7.18 available
 stanford  Stanford AI 

## BLOQUE 2 - ETAPA 1: Inventario de fuentes

Crear registro formal de las 8 fuentes con metadatos completos.

In [58]:
# =============================================================================
# BLOQUE 2 - ETAPA 1: Inventario de fuentes
# =============================================================================

inventario_fuentes = []

for src_id, info in SOURCES.items():
    path = info["file"]
    exists = path.exists()
    
    # Detectar diccionario y auditoria
    has_dict = False
    has_audit = False
    n_tables = 0
    
    if exists and str(path).endswith('.xlsx'):
        try:
            xl = pd.ExcelFile(path, engine='openpyxl')
            sheets = xl.sheet_names
            n_tables = len(sheets)
            has_dict = any('dict' in s.lower() or 'dicc' in s.lower() for s in sheets)
            has_audit = any('audit' in s.lower() for s in sheets)
        except:
            pass
    elif exists and str(path).endswith('.csv'):
        n_tables = 1
    
    # Tamano en MB
    size_mb = round(path.stat().st_size / (1024*1024), 2) if exists else None
    
    inventario_fuentes.append({
        "source_id": src_id,
        "source_name": info["name"],
        "organization": info["organization"],
        "canonical_file": str(path.relative_to(ROOT)),
        "format": info["format"],
        "n_tables_detected": n_tables,
        "primary_table": info["primary_table"],
        "observational_unit_expected": info["obs_unit"],
        "geo_identifier_expected": info["geo_id"],
        "time_coverage_expected": info["time_expected"],
        "extraction_date_available": "",
        "has_dictionary": "SI" if has_dict else "NO",
        "has_audit": "SI" if has_audit else "NO",
        "notes": "",
        "status": "available" if exists else "missing",
        "size_mb": size_mb,
    })

df_fuentes = pd.DataFrame(inventario_fuentes)

print("="*70)
print("INVENTARIO DE FUENTES")
print("="*70)
print(df_fuentes[["source_id", "source_name", "format", "n_tables_detected", 
                  "primary_table", "has_dictionary", "has_audit", "status", "size_mb"]].to_string(index=False))
print()

# Guardar
output_path = OUTPUT_DIR / "inventario_fuentes.csv"
df_fuentes.to_csv(output_path, index=False)
print(f"Guardado: {output_path}")
print(f"Filas: {len(df_fuentes)}")

INVENTARIO DE FUENTES
source_id        source_name format  n_tables_detected                    primary_table has_dictionary has_audit    status  size_mb
     iapp               IAPP   XLSX                  4                            datos             SI        SI available     0.02
microsoft          Microsoft   XLSX                  8                   1_AI_Diffusion             SI        NO available     0.03
   oxford    Oxford Insights   XLSX                  9                      Consolidado             SI        NO available     0.92
       wb World Bank WDI/WGI   XLSX                 10                  MATRIZ_COMPLETA             SI        SI available     1.04
     wipo           WIPO GII   XLSX                  6                       MATRIZ_GII             SI        SI available     7.18
 stanford  Stanford AI Index    CSV                  1 stanford_ai_index_2026_unificado             NO        NO available     4.14
     oecd               OECD   XLSX                 13

## BLOQUE 3 - ETAPA 2: Inventario de tablas

Listar cada hoja/tabla relevante dentro de cada fuente.

In [59]:
# =============================================================================
# BLOQUE 3 - ETAPA 2: Inventario de tablas
# =============================================================================

inventario_tablas = []

for src_id, info in SOURCES.items():
    path = info["file"]
    if not path.exists():
        continue
    
    try:
        if str(path).endswith('.csv'):
            # Para CSV: una sola tabla
            df = pd.read_csv(path, nrows=1000)
            n_rows = sum(1 for _ in open(path, 'r', encoding='utf-8', errors='replace')) - 1
            n_cols = len(df.columns)
            
            is_dict = any(k in str(c).lower() for c in df.columns for k in ['variable', 'descripcion', 'tipo', 'description'])
            is_audit = any(k in str(c).lower() for c in df.columns for k in ['sha256', 'archivo_origen', 'fecha_extraccion', 'audit'])
            is_primary = not is_dict and not is_audit and len(df) > 1
            
            inventario_tablas.append({
                "source_id": src_id,
                "table_id": path.stem,
                "file_path": str(path.relative_to(ROOT)),
                "sheet_or_table_name": path.stem,
                "n_rows": n_rows,
                "n_cols": n_cols,
                "first_columns": " | ".join([str(c) for c in list(df.columns)[:5]]),
                "observational_unit_inferred": info["obs_unit"],
                "geo_col_candidates": " | ".join(detect_geo_columns(df.columns)),
                "time_col_candidates": " | ".join(detect_time_columns(df.columns)),
                "is_dictionary": "SI" if is_dict else "NO",
                "is_audit": "SI" if is_audit else "NO",
                "is_primary_data": "SI" if is_primary else "NO",
                "notes": "",
            })
        else:
            # Para XLSX: listar todas las hojas
            xl = pd.ExcelFile(path, engine='openpyxl')
            for sheet in xl.sheet_names:
                try:
                    df = pd.read_excel(path, sheet_name=sheet, nrows=1000, engine='openpyxl')
                    n_rows = len(pd.read_excel(path, sheet_name=sheet, engine='openpyxl'))
                    n_cols = len(df.columns)
                    
                    is_dict = any(k in str(c).lower() for c in df.columns for k in ['variable', 'descripcion', 'tipo', 'description', 'dict'])
                    is_audit = any(k in str(c).lower() for c in df.columns for k in ['sha256', 'archivo_origen', 'fecha_extraccion', 'audit', 'auditoria'])
                    is_primary = not is_dict and not is_audit and len(df) > 1 and any(k in str(c).lower() for c in df.columns for k in ['iso', 'country', 'pais', 'economy', 'geo'])
                    
                    # Si no tiene geo pero tiene numeros, puede ser primaria tambien
                    if not is_primary and not is_dict and not is_audit and len(df) > 1:
                        numeric_cols = sum(1 for c in df.columns if pd.api.types.is_numeric_dtype(df[c]))
                        if numeric_cols > 0:
                            is_primary = True
                    
                    inventario_tablas.append({
                        "source_id": src_id,
                        "table_id": sheet,
                        "file_path": str(path.relative_to(ROOT)),
                        "sheet_or_table_name": sheet,
                        "n_rows": n_rows,
                        "n_cols": n_cols,
                        "first_columns": " | ".join([str(c) for c in list(df.columns)[:5]]),
                        "observational_unit_inferred": info["obs_unit"] if is_primary else "auxiliar",
                        "geo_col_candidates": " | ".join(detect_geo_columns(df.columns)),
                        "time_col_candidates": " | ".join(detect_time_columns(df.columns)),
                        "is_dictionary": "SI" if is_dict else "NO",
                        "is_audit": "SI" if is_audit else "NO",
                        "is_primary_data": "SI" if is_primary else "NO",
                        "notes": "",
                    })
                except Exception as e:
                    inventario_tablas.append({
                        "source_id": src_id,
                        "table_id": sheet,
                        "file_path": str(path.relative_to(ROOT)),
                        "sheet_or_table_name": sheet,
                        "n_rows": -1,
                        "n_cols": -1,
                        "first_columns": f"ERROR: {str(e)}",
                        "observational_unit_inferred": "unknown",
                        "geo_col_candidates": "",
                        "time_col_candidates": "",
                        "is_dictionary": "NO",
                        "is_audit": "NO",
                        "is_primary_data": "NO",
                        "notes": str(e),
                    })
    except Exception as e:
        print(f"Error procesando {src_id}: {e}")

df_tablas = pd.DataFrame(inventario_tablas)

print("="*70)
print("INVENTARIO DE TABLAS")
print("="*70)
print(df_tablas[["source_id", "table_id", "n_rows", "n_cols", "is_primary_data", "is_dictionary", "is_audit"]].to_string(index=False))
print()
print(f"Total tablas: {len(df_tablas)}")
print(f"Tablas primarias: {len(df_tablas[df_tablas['is_primary_data']=='SI'])}")
print(f"Diccionarios: {len(df_tablas[df_tablas['is_dictionary']=='SI'])}")
print(f"Auditorias: {len(df_tablas[df_tablas['is_audit']=='SI'])}")

# Guardar
output_path = OUTPUT_DIR / "inventario_tablas.csv"
df_tablas.to_csv(output_path, index=False)
print(f"Guardado: {output_path}")

INVENTARIO DE TABLAS
source_id                         table_id  n_rows  n_cols is_primary_data is_dictionary is_audit
     iapp                            datos      29      24              SI            NO       NO
     iapp                      diccionario      24       6              NO            SI       NO
     iapp                      metodologia      13       2              NO            NO       NO
     iapp                        auditoria      20       7              NO            NO       SI
microsoft                         0_README      20       2              NO            NO       NO
microsoft          0_Diccionario_Variables      25      11              NO            SI       NO
microsoft                   1_AI_Diffusion     151       5              SI            NO       NO
microsoft              2_AI_Frontier_Index      13       4              SI            NO       NO
microsoft           3_Data_Center_Capacity      17       3              SI            NO       NO

## BLOQUE 4 - ETAPA 3: Auditoria de esquema

Inventariar todas las columnas de las tablas primarias con tipos, completitud y problemas.

In [60]:
# =============================================================================
# BLOQUE 4 - ETAPA 3: Auditoria de esquema
# =============================================================================

inventario_variables = []
all_issues = []

def add_duplicate_quality_issues(df, source_id, table_id, all_issues):
    """Detecta duplicados sin marcar como problema la repeticion esperada de paneles/formatos long."""
    exact_dups = int(df.duplicated().sum())
    if exact_dups > 0:
        all_issues.append({
            "issue_id": f"{source_id}_{table_id}_DUPROW{len(all_issues)+1:04d}",
            "source_id": source_id, "table_id": table_id, "variable": "__row__", "entity": "",
            "issue_type": "duplicate_row", "severity": "media",
            "description": "Filas completas duplicadas",
            "evidence": f"{exact_dups} filas duplicadas exactas",
            "recommended_action": "Verificar si son repeticiones reales o artefactos de union",
            "phase_to_resolve": "wrangling",
        })

    geo_cols = detect_geo_columns(df.columns)
    if not geo_cols:
        return

    # Solo evaluar duplicado por entidad en tablas que parecen cross-section.
    # En paneles o long format, repetir pais por ano/variable/figura/tarea es esperado.
    structural_keywords = ['year', 'date', 'fecha', 'period', 'fig_id', 'variable', 'indicator', 'cluster', 'task', 'platform']
    has_long_or_time_structure = any(any(k in str(c).lower() for k in structural_keywords) for c in df.columns)
    if has_long_or_time_structure:
        return

    geo_col = geo_cols[0]
    dups = df[df.duplicated(subset=[geo_col], keep=False)]
    if len(dups) > 0:
        for val in dups[geo_col].dropna().unique():
            count = len(dups[dups[geo_col] == val])
            all_issues.append({
                "issue_id": f"{source_id}_{table_id}_DUPENT{len(all_issues)+1:04d}",
                "source_id": source_id, "table_id": table_id, "variable": geo_col,
                "entity": str(val), "issue_type": "duplicate_entity",
                "severity": "media", "description": f"Entidad duplicada en tabla cross-section: {val}",
                "evidence": f"{count} filas para la misma entidad", "recommended_action": "Verificar llave geografica y deduplicar si corresponde",
                "phase_to_resolve": "wrangling",
            })

for src_id, info in SOURCES.items():
    path = info["file"]
    if not path.exists():
        continue
    
    print(f"Procesando {src_id}...")
    
    try:
        if str(path).endswith('.csv'):
            df = pd.read_csv(path, nrows=2000)
            table_id = path.stem
            
            for pos, col in enumerate(df.columns):
                series = df[col]
                n_non_null = series.notna().sum()
                n_null = len(series) - n_non_null
                pct_complete = round(n_non_null / len(series) * 100, 2) if len(series) > 0 else 0
                n_unique = series.nunique(dropna=True)
                inferred = infer_column_type(series)
                
                # Semantic role
                semantic = "unknown"
                if inferred == "geo_identifier": semantic = "geografia"
                elif inferred == "time": semantic = "tiempo"
                elif inferred in ["identifier", "numeric_integer", "numeric_continuous", "binary", "ordinal"]:
                    semantic = "medida"
                elif inferred in ["categorical", "text_short", "text_long"]:
                    semantic = "texto"
                elif inferred == "metadata": semantic = "metadata"
                
                # Example values
                examples = series.dropna().head(3).tolist()
                example_str = " | ".join([str(v)[:50] for v in examples])
                
                # Min/max para numericas
                min_val = None
                max_val = None
                if inferred in ["numeric_continuous", "numeric_integer", "ordinal"]:
                    numeric = pd.to_numeric(series, errors='coerce')
                    if numeric.notna().any():
                        min_val = numeric.min()
                        max_val = numeric.max()
                
                # Quality notes
                quality_notes = []
                if pct_complete == 0: quality_notes.append("100% vacia")
                elif pct_complete < 10: quality_notes.append(f"<10% datos ({pct_complete}%)")
                if n_unique == 1 and n_non_null > 0: quality_notes.append("Constante")
                
                # Detectar porcentaje como texto
                if inferred == "text_short" or inferred == "unknown":
                    if series.astype(str).str.contains('%').any():
                        quality_notes.append("Contiene valores con %")
                
                # Detectar numeros como texto
                if inferred in ["text_short", "unknown", "categorical"]:
                    numeric_test = pd.to_numeric(series.dropna().head(100), errors='coerce')
                    if numeric_test.notna().sum() / len(numeric_test) > 0.5:
                        quality_notes.append("Posiblemente numerica como texto")
                
                inventario_variables.append({
                    "source_id": src_id,
                    "table_id": table_id,
                    "original_variable": str(col),
                    "suggested_variable_name": f"{PREFIXES.get(src_id, '')}{str(col).lower().replace(' ', '_').replace('-', '_')}",
                    "position": pos,
                    "inferred_type": inferred,
                    "semantic_role_raw": semantic,
                    "n_non_null": int(n_non_null),
                    "n_null": int(n_null),
                    "pct_complete": pct_complete,
                    "n_unique": int(n_unique),
                    "is_empty": 1 if pct_complete == 0 else 0,
                    "is_constant": 1 if n_unique == 1 and n_non_null > 0 else 0,
                    "is_numeric_like": 1 if inferred in ["numeric_continuous", "numeric_integer", "ordinal", "binary"] else 0,
                    "is_percent_like": 1 if any('%' in str(v) for v in series.dropna().head(10)) else 0,
                    "is_money_like": 1 if any(k in str(col).lower() for k in ['usd', 'gdp', 'income', 'spending']) else 0,
                    "is_rank_like": 1 if 'rank' in str(col).lower() else 0,
                    "is_year_like": 1 if inferred == "time" else 0,
                    "min_value": min_val,
                    "max_value": max_val,
                    "example_values": example_str,
                    "quality_notes": "; ".join(quality_notes) if quality_notes else "",
                })
            
            # Quality issues de esta tabla
            table_issues = detect_quality_issues(df, src_id, table_id)
            all_issues.extend(table_issues)
            
        else:
            # XLSX: procesar solo tablas primarias
            xl = pd.ExcelFile(path, engine='openpyxl')
            for sheet in xl.sheet_names:
                try:
                    df = pd.read_excel(path, sheet_name=sheet, nrows=2000, engine='openpyxl')
                    n_rows_full = len(pd.read_excel(path, sheet_name=sheet, engine='openpyxl'))
                    
                    # Solo auditar tablas con datos (no diccionarios ni auditorias)
                    is_dict = any(k in str(c).lower() for c in df.columns for k in ['variable', 'descripcion', 'tipo', 'description', 'dict'])
                    is_audit = any(k in str(c).lower() for c in df.columns for k in ['sha256', 'archivo_origen', 'fecha_extraccion', 'audit'])
                    
                    if is_dict or is_audit or len(df) <= 1:
                        continue
                    
                    for pos, col in enumerate(df.columns):
                        series = df[col]
                        n_non_null = series.notna().sum()
                        n_null = n_rows_full - n_non_null
                        pct_complete = round(n_non_null / n_rows_full * 100, 2) if n_rows_full > 0 else 0
                        n_unique = series.nunique(dropna=True)
                        inferred = infer_column_type(series)
                        
                        semantic = "unknown"
                        if inferred == "geo_identifier": semantic = "geografia"
                        elif inferred == "time": semantic = "tiempo"
                        elif inferred in ["identifier", "numeric_integer", "numeric_continuous", "binary", "ordinal"]:
                            semantic = "medida"
                        elif inferred in ["categorical", "text_short", "text_long"]:
                            semantic = "texto"
                        elif inferred == "metadata": semantic = "metadata"
                        
                        examples = series.dropna().head(3).tolist()
                        example_str = " | ".join([str(v)[:50] for v in examples])
                        
                        min_val = None
                        max_val = None
                        if inferred in ["numeric_continuous", "numeric_integer", "ordinal"]:
                            numeric = pd.to_numeric(series, errors='coerce')
                            if numeric.notna().any():
                                min_val = numeric.min()
                                max_val = numeric.max()
                        
                        quality_notes = []
                        if pct_complete == 0: quality_notes.append("100% vacia")
                        elif pct_complete < 10: quality_notes.append(f"<10% datos ({pct_complete}%)")
                        if n_unique == 1 and n_non_null > 0: quality_notes.append("Constante")
                        if series.astype(str).str.contains('%').any():
                            quality_notes.append("Contiene valores con %")
                        
                        numeric_test = pd.to_numeric(series.dropna().head(100), errors='coerce')
                        if numeric_test.notna().sum() / max(len(numeric_test), 1) > 0.5 and inferred not in ["numeric_continuous", "numeric_integer", "ordinal", "binary"]:
                            quality_notes.append("Posiblemente numerica como texto")
                        
                        inventario_variables.append({
                            "source_id": src_id,
                            "table_id": sheet,
                            "original_variable": str(col),
                            "suggested_variable_name": f"{PREFIXES.get(src_id, '')}{str(col).lower().replace(' ', '_').replace('-', '_')}",
                            "position": pos,
                            "inferred_type": inferred,
                            "semantic_role_raw": semantic,
                            "n_non_null": int(n_non_null),
                            "n_null": int(n_null),
                            "pct_complete": pct_complete,
                            "n_unique": int(n_unique),
                            "is_empty": 1 if pct_complete == 0 else 0,
                            "is_constant": 1 if n_unique == 1 and n_non_null > 0 else 0,
                            "is_numeric_like": 1 if inferred in ["numeric_continuous", "numeric_integer", "ordinal", "binary"] else 0,
                            "is_percent_like": 1 if any('%' in str(v) for v in series.dropna().head(10)) else 0,
                            "is_money_like": 1 if any(k in str(col).lower() for k in ['usd', 'gdp', 'income', 'spending']) else 0,
                            "is_rank_like": 1 if 'rank' in str(col).lower() else 0,
                            "is_year_like": 1 if inferred == "time" else 0,
                            "min_value": min_val,
                            "max_value": max_val,
                            "example_values": example_str,
                            "quality_notes": "; ".join(quality_notes) if quality_notes else "",
                        })
                    
                    # Quality issues
                    df_full_for_issues = pd.read_excel(path, sheet_name=sheet, engine='openpyxl')
                    table_issues = detect_quality_issues(df_full_for_issues, src_id, sheet)
                    all_issues.extend(table_issues)
                    
                except Exception as e:
                    print(f"  Error en hoja {sheet}: {e}")
                    
    except Exception as e:
        print(f"Error procesando {src_id}: {e}")

df_variables = pd.DataFrame(inventario_variables)

print()
print("="*70)
print("AUDITORIA DE ESQUEMA - RESUMEN")
print("="*70)
print(f"Total variables auditadas: {len(df_variables)}")
print(f"Variables vacias (100%): {len(df_variables[df_variables['is_empty']==1])}")
print(f"Variables constantes: {len(df_variables[df_variables['is_constant']==1])}")
print(f"Variables con >90% faltantes: {len(df_variables[df_variables['pct_complete']<10])}")
print()

# Resumen por fuente
print("Resumen por fuente:")
resumen = df_variables.groupby('source_id').agg({
    'original_variable': 'count',
    'is_empty': 'sum',
    'is_constant': 'sum',
    'pct_complete': 'mean'
}).round(2)
resumen.columns = ['n_vars', 'vacias', 'constantes', 'pct_complete_prom']
print(resumen.to_string())

# Guardar
output_path = OUTPUT_DIR / "inventario_variables.csv"
df_variables.to_csv(output_path, index=False)
print(f"Guardado: {output_path}")

# Guardar issues parciales
df_issues_partial = pd.DataFrame(all_issues)
if len(df_issues_partial) > 0:
    print(f"Issues detectados hasta ahora: {len(df_issues_partial)}")

Procesando iapp...
Procesando microsoft...
Procesando oxford...
Procesando wb...
Procesando wipo...
Procesando stanford...
Procesando oecd...
Procesando anthropic...

AUDITORIA DE ESQUEMA - RESUMEN
Total variables auditadas: 3311
Variables vacias (100%): 21
Variables constantes: 193
Variables con >90% faltantes: 186

Resumen por fuente:
           n_vars  vacias  constantes  pct_complete_prom
source_id                                               
anthropic     331       2          29              34.31
iapp           26       0           2             100.00
microsoft      26       1           1              71.67
oecd           48       0           1              85.82
oxford        284      17          15              59.85
stanford        9       0           1              88.67
wb            602       0           2              84.39
wipo         1985       1         142              84.81
Guardado: /home/pablo/Research_LeyIA_DataScience/Research_AI_law/outputs/eda_preliminar/inv

## BLOQUE 5 - ETAPA 4: Auditoria geografica

Evaluar integrabilidad por pais y separar paises de territorios, regiones y agregados.

In [61]:
# =============================================================================
# BLOQUE 5 - ETAPA 4: Auditoria geografica
# =============================================================================

entidades_no_pais = []

for src_id, info in SOURCES.items():
    path = info["file"]
    if not path.exists():
        continue
    print(f"Procesando geografia de {src_id}...")
    try:
        if str(path).endswith('.csv'):
            df = pd.read_csv(path)
            geo_cols = detect_geo_columns(df.columns)
            if not geo_cols:
                print(f"  No se detectaron columnas geograficas")
                continue
            geo_col = geo_cols[0]
            for idx, row in df.iterrows():
                raw_code = row.get(geo_col, "")
                raw_name = ""
                name_cols = [c for c in df.columns if any(k in str(c).lower() for k in ['name', 'pais', 'country', 'economy'])]
                if name_cols:
                    raw_name = row.get(name_cols[0], "")
                entity_type = classify_entity(raw_code, raw_name)
                if entity_type != "country_iso3":
                    entidades_no_pais.append({
                        "source_id": src_id, "table_id": path.stem,
                        "raw_entity_name": str(raw_name) if raw_name else str(raw_code),
                        "raw_geo_code": str(raw_code) if raw_code else "",
                        "entity_type_inferred": entity_type,
                        "reason": f"Clasificado como {entity_type}",
                        "recommendation": "Revisar" if entity_type == "unknown" else "Mapear manualmente",
                    })
        else:
            xl = pd.ExcelFile(path, engine='openpyxl')
            for sheet in xl.sheet_names:
                try:
                    df = pd.read_excel(path, sheet_name=sheet, nrows=5000, engine='openpyxl')
                    geo_cols = detect_geo_columns(df.columns)
                    if not geo_cols:
                        continue
                    geo_col = geo_cols[0]
                    for idx, row in df.iterrows():
                        raw_code = row.get(geo_col, "")
                        raw_name = ""
                        name_cols = [c for c in df.columns if any(k in str(c).lower() for k in ['name', 'pais', 'country', 'economy'])]
                        if name_cols:
                            raw_name = row.get(name_cols[0], "")
                        entity_type = classify_entity(raw_code, raw_name)
                        if entity_type != "country_iso3":
                            entidades_no_pais.append({
                                "source_id": src_id, "table_id": sheet,
                                "raw_entity_name": str(raw_name) if pd.notna(raw_name) and raw_name else str(raw_code),
                                "raw_geo_code": str(raw_code) if pd.notna(raw_code) else "",
                                "entity_type_inferred": entity_type,
                                "reason": f"Clasificado como {entity_type}",
                                "recommendation": "Revisar" if entity_type == "unknown" else "Mapear manualmente",
                            })
                except:
                    pass
    except Exception as e:
        print(f"Error en {src_id}: {e}")

df_entidades = pd.DataFrame(entidades_no_pais)
if len(df_entidades) > 0:
    df_entidades = df_entidades.drop_duplicates().reset_index(drop=True)
print()
print("="*70)
print("ENTIDADES NO PAIS O SIN ISO3")
print("="*70)
if len(df_entidades) > 0:
    print(df_entidades.groupby(['source_id', 'entity_type_inferred']).size().to_string())
    print(f"Total entidades no-pais: {len(df_entidades)}")
    output_path = OUTPUT_DIR / "entidades_no_pais_o_sin_iso3.csv"
    df_entidades.to_csv(output_path, index=False)
    print(f"Guardado: {output_path}")
else:
    print("No se detectaron entidades no-pais")

Procesando geografia de iapp...
Procesando geografia de microsoft...
Procesando geografia de oxford...
Procesando geografia de wb...
Procesando geografia de wipo...
Procesando geografia de stanford...
Procesando geografia de oecd...
Procesando geografia de anthropic...

ENTIDADES NO PAIS O SIN ISO3
source_id  entity_type_inferred 
anthropic  global                      4
           region                     38
           subnational                53
           territory_iso3             45
           unknown                  2919
iapp       organization_or_group       1
           territory_iso3              1
microsoft  region                      4
           subnational                 2
           unknown                   175
oecd       organization_or_group       3
           region                      2
           territory_iso3              4
           unknown                    17
stanford   region                     30
           subnational                 2
           

In [62]:
# =============================================================================
# COBERTURA PAIS X FUENTE
# =============================================================================
all_countries = {}
for src_id, info in SOURCES.items():
    path = info["file"]
    if not path.exists():
        continue
    try:
        if str(path).endswith('.csv'):
            df = pd.read_csv(path)
            geo_cols = detect_geo_columns(df.columns)
            if geo_cols:
                geo_col = geo_cols[0]
                for val in df[geo_col].dropna().unique():
                    code = str(val).upper().strip()
                    if len(code) == 3 and code.isalpha():
                        name_cols = [c for c in df.columns if any(k in str(c).lower() for k in ['name', 'pais', 'country', 'economy'])]
                        name = ""
                        if name_cols:
                            name = df[df[geo_col] == val][name_cols[0]].iloc[0] if len(df[df[geo_col] == val]) > 0 else ""
                        entity_type = classify_entity(code, name)
                        if entity_type in ["country_iso3", "territory_iso3"]:
                            if code not in all_countries:
                                all_countries[code] = {"sources": set(), "names": set(), "entity_type": entity_type}
                            all_countries[code]["sources"].add(src_id)
                            all_countries[code]["entity_type"] = entity_type if all_countries[code].get("entity_type") != "country_iso3" else "country_iso3"
                            if pd.notna(name) and name != "":
                                all_countries[code]["names"].add(str(name))
        else:
            xl = pd.ExcelFile(path, engine='openpyxl')
            for sheet in xl.sheet_names:
                try:
                    df = pd.read_excel(path, sheet_name=sheet, nrows=5000, engine='openpyxl')
                    geo_cols = detect_geo_columns(df.columns)
                    if not geo_cols:
                        continue
                    geo_col = geo_cols[0]
                    for val in df[geo_col].dropna().unique():
                        code = str(val).upper().strip()
                        if len(code) == 3 and code.isalpha():
                            name_cols = [c for c in df.columns if any(k in str(c).lower() for k in ['name', 'pais', 'country', 'economy'])]
                            name = ""
                            if name_cols:
                                subset = df[df[geo_col] == val]
                                if len(subset) > 0:
                                    name = subset[name_cols[0]].iloc[0]
                            entity_type = classify_entity(code, name)
                            if entity_type in ["country_iso3", "territory_iso3"]:
                                if code not in all_countries:
                                    all_countries[code] = {"sources": set(), "names": set(), "entity_type": entity_type}
                                all_countries[code]["sources"].add(src_id)
                                all_countries[code]["entity_type"] = entity_type if all_countries[code].get("entity_type") != "country_iso3" else "country_iso3"
                                if pd.notna(name) and name != "":
                                    all_countries[code]["names"].add(str(name))
                except:
                    pass
    except Exception as e:
        print(f"Error en {src_id}: {e}")

cobertura_rows = []
for iso3, data in sorted(all_countries.items()):
    row = {"iso3": iso3, "country_name_best_effort": " | ".join(list(data["names"])[:3]), "entity_type": data.get("entity_type", "country_iso3")}
    for src_id in SOURCES.keys():
        row[f"present_{src_id}"] = 1 if src_id in data["sources"] else 0
    row["n_sources_present"] = len(data["sources"])
    row["source_list"] = ", ".join(sorted(data["sources"]))
    row["integration_notes"] = ""
    cobertura_rows.append(row)

df_cobertura = pd.DataFrame(cobertura_rows).sort_values("n_sources_present", ascending=False)
print("="*70)
print("COBERTURA PAIS X FUENTE")
print("="*70)
print(f"Total paises: {len(df_cobertura)}")
print(f"Paises en 7+ fuentes: {len(df_cobertura[df_cobertura['n_sources_present'] >= 7])}")
print(f"Paises en 5+ fuentes: {len(df_cobertura[df_cobertura['n_sources_present'] >= 5])}")
print("\nTop 20 paises con mayor cobertura:")
print(df_cobertura.head(20)[["iso3", "country_name_best_effort", "n_sources_present", "source_list"]].to_string(index=False))
print("\nCobertura por fuente:")
for src_id in SOURCES.keys():
    n = df_cobertura[f"present_{src_id}"].sum()
    print(f"  {src_id}: {int(n)} paises")
output_path = OUTPUT_DIR / "cobertura_pais_fuente.csv"
df_cobertura.to_csv(output_path, index=False)
print(f"Guardado: {output_path}")

COBERTURA PAIS X FUENTE
Total paises: 259
Paises en 7+ fuentes: 26
Paises en 5+ fuentes: 190

Top 20 paises con mayor cobertura:
iso3                                                    country_name_best_effort  n_sources_present                                       source_list
 TUR                                                  Turkey | Turkiye | Türkiye                  7 anthropic, iapp, oecd, oxford, stanford, wb, wipo
 BRA                                                                BRA | Brazil                  7 anthropic, iapp, oecd, oxford, stanford, wb, wipo
 GBR United Kingdom | GBR | United Kingdom of Great Britain and Northern Ireland                  7 anthropic, iapp, oecd, oxford, stanford, wb, wipo
 USA                              United States of America | USA | United States                  7 anthropic, iapp, oecd, oxford, stanford, wb, wipo
 CHN                                                                 CHN | China                  7 anthropic, iapp, oecd

## BLOQUE 6 - ETAPA 5: Auditoria temporal

In [63]:
# =============================================================================
# BLOQUE 6 - ETAPA 5: Auditoria temporal
# =============================================================================
import re as re_mod
temporal_results = []
for src_id, info in SOURCES.items():
    path = info["file"]
    if not path.exists():
        continue
    print(f"Procesando temporalidad de {src_id}...")
    try:
        if str(path).endswith('.csv'):
            df = pd.read_csv(path, nrows=5000)
            time_cols = detect_time_columns(df.columns)
            min_year = None; max_year = None; n_years = 0; temporal_structure = "unknown"
            if time_cols:
                year_col = time_cols[0]
                if year_col in df.columns:
                    years = pd.to_numeric(df[year_col], errors='coerce').dropna()
                    if len(years) > 0:
                        min_year = int(years.min()); max_year = int(years.max()); n_years = int(years.nunique())
                        temporal_structure = "panel_long" if len(df) > len(years.unique()) * 2 else "cross_section"
            else:
                temporal_structure = "cross_section"
            temporal_results.append({
                "source_id": src_id, "table_id": path.stem, "temporal_structure": temporal_structure,
                "year_columns_detected": " | ".join(time_cols), "min_year": min_year, "max_year": max_year,
                "n_years": n_years, "n_entities_by_year_summary": "", "date_columns_detected": "",
                "snapshot_candidate": "SI" if temporal_structure == "panel_long" else "NO",
                "snapshot_rule_candidate": "ultimo_ano_disponible" if temporal_structure == "panel_long" else "",
                "notes": "",
            })
        else:
            xl = pd.ExcelFile(path, engine='openpyxl')
            for sheet in xl.sheet_names:
                try:
                    df = pd.read_excel(path, sheet_name=sheet, nrows=5000, engine='openpyxl')
                    time_cols = detect_time_columns(df.columns)
                    year_in_cols = [c for c in df.columns if re_mod.search(r'_(19|20)\d{2}$', str(c)) or re_mod.search(r'\b(19|20)\d{2}\b', str(c))]
                    min_year = None; max_year = None; n_years = 0; temporal_structure = "unknown"
                    if time_cols and any(c in df.columns for c in time_cols):
                        year_col = [c for c in time_cols if c in df.columns][0]
                        years = pd.to_numeric(df[year_col], errors='coerce').dropna()
                        if len(years) > 0:
                            min_year = int(years.min()); max_year = int(years.max()); n_years = int(years.nunique())
                            temporal_structure = "panel_long"
                    elif year_in_cols:
                        years = []
                        for c in year_in_cols:
                            match = re_mod.search(r'(19|20)\d{2}', str(c))
                            if match: years.append(int(match.group()))
                        if years:
                            min_year = min(years); max_year = max(years); n_years = len(set(years))
                            temporal_structure = "panel_wide"
                    else:
                        temporal_structure = "cross_section"
                    temporal_results.append({
                        "source_id": src_id, "table_id": sheet, "temporal_structure": temporal_structure,
                        "year_columns_detected": " | ".join(time_cols + year_in_cols) if (time_cols or year_in_cols) else "",
                        "min_year": min_year, "max_year": max_year, "n_years": n_years,
                        "n_entities_by_year_summary": "", "date_columns_detected": "",
                        "snapshot_candidate": "SI" if temporal_structure in ["panel_long", "panel_wide"] else "NO",
                        "snapshot_rule_candidate": "ultimo_ano_disponible" if temporal_structure in ["panel_long", "panel_wide"] else "",
                        "notes": "",
                    })
                except Exception as e:
                    print(f"  Error en hoja {sheet}: {e}")
    except Exception as e:
        print(f"Error en {src_id}: {e}")

df_temporal = pd.DataFrame(temporal_results)
print()
print("="*70)
print("AUDITORIA TEMPORAL")
print("="*70)
print(df_temporal[["source_id", "table_id", "temporal_structure", "min_year", "max_year", "n_years", "snapshot_candidate"]].to_string(index=False))
output_path = OUTPUT_DIR / "cobertura_temporal_fuente.csv"
df_temporal.to_csv(output_path, index=False)
print(f"\nGuardado: {output_path}")

df_snapshot_candidates = df_temporal[df_temporal["snapshot_candidate"] == "SI"].copy()
if len(df_snapshot_candidates) > 0:
    df_snapshot_candidates["candidate_reason"] = "Fuente con estructura temporal panel o wide; snapshot solo como decision futura de Fase 3"
    df_snapshot_candidates["is_final_snapshot"] = 0
else:
    df_snapshot_candidates = pd.DataFrame(columns=list(df_temporal.columns) + ["candidate_reason", "is_final_snapshot"])
snapshot_path = OUTPUT_DIR / "variables_candidatas_snapshot.csv"
df_snapshot_candidates.to_csv(snapshot_path, index=False)
print(f"Guardado: {snapshot_path}")

Procesando temporalidad de iapp...
Procesando temporalidad de microsoft...
Procesando temporalidad de oxford...
Procesando temporalidad de wb...
Procesando temporalidad de wipo...
Procesando temporalidad de stanford...
Procesando temporalidad de oecd...
Procesando temporalidad de anthropic...

AUDITORIA TEMPORAL
source_id                         table_id temporal_structure  min_year  max_year  n_years snapshot_candidate
     iapp                            datos      cross_section       NaN       NaN        0                 NO
     iapp                      diccionario      cross_section       NaN       NaN        0                 NO
     iapp                      metodologia      cross_section       NaN       NaN        0                 NO
     iapp                        auditoria      cross_section       NaN       NaN        0                 NO
microsoft                         0_README      cross_section       NaN       NaN        0                 NO
microsoft          0_Dicci

## BLOQUE 7 - ETAPA 6: Completitud

In [64]:
# =============================================================================
# BLOQUE 7 - ETAPA 6: Completitud
# =============================================================================
completitud_rows = []
for src_id, info in SOURCES.items():
    path = info["file"]
    if not path.exists():
        continue
    print(f"Calculando completitud de {src_id}...")
    try:
        if str(path).endswith('.csv'):
            df = pd.read_csv(path, nrows=2000)
            comp = compute_completeness(df)
            n_attrs = comp["n_cols"]
            for idx, row in df.iterrows():
                n_avail = row.notna().sum()
                pct = round(n_avail / n_attrs * 100, 2)
                geo_cols = detect_geo_columns(df.columns)
                iso3 = ""
                if geo_cols and geo_cols[0] in df.columns:
                    iso3 = str(row.get(geo_cols[0], "")).upper().strip()
                completitud_rows.append({
                    "source_id": src_id, "table_id": path.stem, "iso3": iso3 if len(iso3) == 3 and iso3.isalpha() else "",
                    "country_name_best_effort": "", "n_attributes_available": int(n_avail),
                    "n_attributes_total": n_attrs, "pct_attributes_available": pct,
                    "n_years_available": "", "min_year_available": "", "max_year_available": "", "notes": "",
                })
        else:
            xl = pd.ExcelFile(path, engine='openpyxl')
            for sheet in xl.sheet_names:
                try:
                    df = pd.read_excel(path, sheet_name=sheet, nrows=2000, engine='openpyxl')
                    is_dict = any(k in str(c).lower() for c in df.columns for k in ['variable', 'descripcion', 'tipo', 'description', 'dict'])
                    is_audit = any(k in str(c).lower() for c in df.columns for k in ['sha256', 'archivo_origen', 'fecha_extraccion', 'audit'])
                    if is_dict or is_audit or len(df) <= 1:
                        continue
                    comp = compute_completeness(df)
                    n_attrs = comp["n_cols"]
                    for idx, row in df.iterrows():
                        n_avail = row.notna().sum()
                        pct = round(n_avail / n_attrs * 100, 2)
                        geo_cols = detect_geo_columns(df.columns)
                        iso3 = ""
                        if geo_cols and geo_cols[0] in df.columns:
                            iso3 = str(row.get(geo_cols[0], "")).upper().strip()
                        completitud_rows.append({
                            "source_id": src_id, "table_id": sheet, "iso3": iso3 if len(iso3) == 3 and iso3.isalpha() else "",
                            "country_name_best_effort": "", "n_attributes_available": int(n_avail),
                            "n_attributes_total": n_attrs, "pct_attributes_available": pct,
                            "n_years_available": "", "min_year_available": "", "max_year_available": "", "notes": "",
                        })
                except:
                    pass
    except Exception as e:
        print(f"Error en {src_id}: {e}")

df_completitud = pd.DataFrame(completitud_rows)
if len(df_completitud) > 0:
    group_cols = ["source_id", "table_id", "iso3", "country_name_best_effort"]
    df_completitud = (
        df_completitud.fillna("")
        .groupby(group_cols, dropna=False, as_index=False)
        .agg(
            n_attributes_available=("n_attributes_available", "sum"),
            n_attributes_total=("n_attributes_total", "sum"),
            n_years_available=("n_years_available", "first"),
            min_year_available=("min_year_available", "first"),
            max_year_available=("max_year_available", "first"),
            notes=("notes", "first"),
        )
    )
    df_completitud["pct_attributes_available"] = (
        df_completitud["n_attributes_available"] / df_completitud["n_attributes_total"] * 100
    ).round(2)
    ordered_cols = [
        "source_id", "table_id", "iso3", "country_name_best_effort", "n_attributes_available",
        "n_attributes_total", "pct_attributes_available", "n_years_available", "min_year_available",
        "max_year_available", "notes",
    ]
    df_completitud = df_completitud[ordered_cols]
print()
print("="*70)
print("COMPLETITUD - RESUMEN")
print("="*70)
if len(df_completitud) > 0:
    print(f"Total filas auditadas: {len(df_completitud)}")
    print(f"Promedio % disponible: {df_completitud['pct_attributes_available'].mean():.1f}%")
    print(f"Mediana % disponible: {df_completitud['pct_attributes_available'].median():.1f}%")
    print()
    resumen_comp = df_completitud.groupby('source_id')['pct_attributes_available'].agg(['mean', 'median', 'min', 'max']).round(1)
    print("Completitud por fuente:")
    print(resumen_comp.to_string())
    output_path = OUTPUT_DIR / "cobertura_pais_atributo.csv"
    df_completitud.to_csv(output_path, index=False)
    print(f"\nGuardado: {output_path}")
else:
    print("No se pudieron calcular metricas")

Calculando completitud de iapp...
Calculando completitud de microsoft...
Calculando completitud de oxford...
Calculando completitud de wb...
Calculando completitud de wipo...
Calculando completitud de stanford...
Calculando completitud de oecd...
Calculando completitud de anthropic...

COMPLETITUD - RESUMEN
Total filas auditadas: 4358
Promedio % disponible: 82.5%
Mediana % disponible: 95.0%

Completitud por fuente:
            mean  median    min    max
source_id                             
anthropic   80.4    83.3   16.2  100.0
iapp       100.0   100.0  100.0  100.0
microsoft   51.1    56.5    0.0  100.0
oecd        97.0   100.0   30.0  100.0
oxford      77.8    94.1   20.9  100.0
stanford    88.9    88.9   88.7   88.9
wb          80.7    95.0    3.1  100.0
wipo        80.8    88.1   14.4   99.7

Guardado: /home/pablo/Research_LeyIA_DataScience/Research_AI_law/outputs/eda_preliminar/cobertura_pais_atributo.csv


## BLOQUE 8 - ETAPA 7: Perfil estadistico univariado

Describir variables numericas y categoricas sin relacionar con hipotesis regulatorias.

In [65]:
# =============================================================================
# BLOQUE 8 - ETAPA 7: Perfil estadistico univariado
# =============================================================================

perfil_results = []

for src_id, info in SOURCES.items():
    path = info["file"]
    if not path.exists():
        continue
    print(f"Perfilando {src_id}...")
    try:
        if str(path).endswith('.csv'):
            df = pd.read_csv(path, nrows=2000)
            for col in df.columns:
                series = df[col]
                n_non_null = series.notna().sum()
                if n_non_null == 0:
                    continue
                inferred = infer_column_type(series)
                if inferred in ["numeric_continuous", "numeric_integer", "ordinal"]:
                    numeric = pd.to_numeric(series, errors='coerce')
                    numeric_clean = numeric.dropna()
                    if len(numeric_clean) > 0:
                        perfil_results.append({
                            "source_id": src_id, "table_id": path.stem, "variable": str(col),
                            "type": "numeric", "count": int(len(numeric_clean)),
                            "missing": int(len(series) - n_non_null), "mean": round(numeric_clean.mean(), 4),
                            "median": round(numeric_clean.median(), 4), "std": round(numeric_clean.std(), 4),
                            "min": round(numeric_clean.min(), 4), "max": round(numeric_clean.max(), 4),
                            "p01": round(numeric_clean.quantile(0.01), 4) if len(numeric_clean) > 1 else None,
                            "p05": round(numeric_clean.quantile(0.05), 4) if len(numeric_clean) > 1 else None,
                            "p25": round(numeric_clean.quantile(0.25), 4) if len(numeric_clean) > 1 else None,
                            "p75": round(numeric_clean.quantile(0.75), 4) if len(numeric_clean) > 1 else None,
                            "p95": round(numeric_clean.quantile(0.95), 4) if len(numeric_clean) > 1 else None,
                            "p99": round(numeric_clean.quantile(0.99), 4) if len(numeric_clean) > 1 else None,
                            "n_zero": int((numeric_clean == 0).sum()),
                            "n_negative": int((numeric_clean < 0).sum()),
                            "skewness": round(numeric_clean.skew(), 4) if len(numeric_clean) > 2 else None,
                            "kurtosis": round(numeric_clean.kurtosis(), 4) if len(numeric_clean) > 2 else None,
                            "iqr": round(numeric_clean.quantile(0.75) - numeric_clean.quantile(0.25), 4) if len(numeric_clean) > 1 else None,
                        })
                elif inferred in ["categorical", "binary"]:
                    top_vals = series.value_counts().head(5)
                    perfil_results.append({
                        "source_id": src_id, "table_id": path.stem, "variable": str(col),
                        "type": "categorical", "count": int(n_non_null),
                        "missing": int(len(series) - n_non_null), "n_categories": int(series.nunique(dropna=True)),
                        "modal_category": str(top_vals.index[0]) if len(top_vals) > 0 else "",
                        "modal_frequency": int(top_vals.iloc[0]) if len(top_vals) > 0 else 0,
                        "top_5_categories": " | ".join([str(v) for v in top_vals.index[:5]]),
                        "mean": None, "median": None, "std": None, "min": None, "max": None,
                        "p01": None, "p05": None, "p25": None, "p75": None, "p95": None, "p99": None,
                        "n_zero": None, "n_negative": None, "skewness": None, "kurtosis": None, "iqr": None,
                    })
        else:
            xl = pd.ExcelFile(path, engine='openpyxl')
            for sheet in xl.sheet_names:
                try:
                    df = pd.read_excel(path, sheet_name=sheet, nrows=2000, engine='openpyxl')
                    is_dict = any(k in str(c).lower() for c in df.columns for k in ['variable', 'descripcion', 'tipo', 'description', 'dict'])
                    is_audit = any(k in str(c).lower() for c in df.columns for k in ['sha256', 'archivo_origen', 'fecha_extraccion', 'audit'])
                    if is_dict or is_audit or len(df) <= 1:
                        continue
                    for col in df.columns:
                        series = df[col]
                        n_non_null = series.notna().sum()
                        if n_non_null == 0:
                            continue
                        inferred = infer_column_type(series)
                        if inferred in ["numeric_continuous", "numeric_integer", "ordinal"]:
                            numeric = pd.to_numeric(series, errors='coerce')
                            numeric_clean = numeric.dropna()
                            if len(numeric_clean) > 0:
                                perfil_results.append({
                                    "source_id": src_id, "table_id": sheet, "variable": str(col),
                                    "type": "numeric", "count": int(len(numeric_clean)),
                                    "missing": int(len(series) - n_non_null), "mean": round(numeric_clean.mean(), 4),
                                    "median": round(numeric_clean.median(), 4), "std": round(numeric_clean.std(), 4),
                                    "min": round(numeric_clean.min(), 4), "max": round(numeric_clean.max(), 4),
                                    "p01": round(numeric_clean.quantile(0.01), 4) if len(numeric_clean) > 1 else None,
                                    "p05": round(numeric_clean.quantile(0.05), 4) if len(numeric_clean) > 1 else None,
                                    "p25": round(numeric_clean.quantile(0.25), 4) if len(numeric_clean) > 1 else None,
                                    "p75": round(numeric_clean.quantile(0.75), 4) if len(numeric_clean) > 1 else None,
                                    "p95": round(numeric_clean.quantile(0.95), 4) if len(numeric_clean) > 1 else None,
                                    "p99": round(numeric_clean.quantile(0.99), 4) if len(numeric_clean) > 1 else None,
                                    "n_zero": int((numeric_clean == 0).sum()),
                                    "n_negative": int((numeric_clean < 0).sum()),
                                    "skewness": round(numeric_clean.skew(), 4) if len(numeric_clean) > 2 else None,
                                    "kurtosis": round(numeric_clean.kurtosis(), 4) if len(numeric_clean) > 2 else None,
                                    "iqr": round(numeric_clean.quantile(0.75) - numeric_clean.quantile(0.25), 4) if len(numeric_clean) > 1 else None,
                                })
                        elif inferred in ["categorical", "binary"]:
                            top_vals = series.value_counts().head(5)
                            perfil_results.append({
                                "source_id": src_id, "table_id": sheet, "variable": str(col),
                                "type": "categorical", "count": int(n_non_null),
                                "missing": int(len(series) - n_non_null), "n_categories": int(series.nunique(dropna=True)),
                                "modal_category": str(top_vals.index[0]) if len(top_vals) > 0 else "",
                                "modal_frequency": int(top_vals.iloc[0]) if len(top_vals) > 0 else 0,
                                "top_5_categories": " | ".join([str(v) for v in top_vals.index[:5]]),
                                "mean": None, "median": None, "std": None, "min": None, "max": None,
                                "p01": None, "p05": None, "p25": None, "p75": None, "p95": None, "p99": None,
                                "n_zero": None, "n_negative": None, "skewness": None, "kurtosis": None, "iqr": None,
                            })
                except Exception as e:
                    print(f"  Error en hoja {sheet}: {e}")
    except Exception as e:
        print(f"Error en {src_id}: {e}")

df_perfil = pd.DataFrame(perfil_results)
print()
print("="*70)
print("PERFIL UNIVARIADO - RESUMEN")
print("="*70)
print(f"Variables numericas perfiladas: {len(df_perfil[df_perfil['type']=='numeric'])}")
print(f"Variables categoricas perfiladas: {len(df_perfil[df_perfil['type']=='categorical'])}")
if len(df_perfil) > 0:
    print("\nTop 10 variables numericas por n de observaciones:")
    top_num = df_perfil[df_perfil['type']=='numeric'].nlargest(10, 'count')[['source_id', 'variable', 'count', 'mean', 'std', 'min', 'max']]
    print(top_num.to_string(index=False))

Perfilando iapp...
Perfilando microsoft...
Perfilando oxford...
Perfilando wb...
Perfilando wipo...
Perfilando stanford...
Perfilando oecd...
Perfilando anthropic...

PERFIL UNIVARIADO - RESUMEN
Variables numericas perfiladas: 3019
Variables categoricas perfiladas: 125

Top 10 variables numericas por n de observaciones:
source_id           variable  count         mean          std     min          max
 stanford            chapter   2000 1.000000e+00 0.000000e+00  1.0000 1.000000e+00
 stanford              value   2000 1.040688e+12 1.887263e+13  0.0000 7.987200e+14
     oecd  oecd_indigo_score   2000 8.400000e-03 2.180000e-02  0.0000 1.142000e-01
     oecd oecd_gbard_usd_ppp   2000 1.200688e+04 2.611170e+04 12.2759 1.993420e+05
anthropic            task_id   2000 1.118910e+04 6.864010e+03  1.0000 2.072300e+04
anthropic       _origin_line   2000 1.001500e+03 5.774946e+02  2.0000 2.001000e+03
anthropic      feedback_loop   2000 1.910000e-02 5.750000e-02  0.0000 5.069000e-01
anthropic     

## BLOQUE 9 - ETAPA 8: Diagnostico de calidad

Crear bitacora accionable de todos los problemas detectados.

In [66]:
# =============================================================================
# BLOQUE 9 - ETAPA 8: Diagnostico de calidad
# =============================================================================

# Los issues ya fueron parcialmente detectados en el Bloque 4
# Aqui consolidamos todos los issues y agregamos mas detecciones

all_issues = []

for src_id, info in SOURCES.items():
    path = info["file"]
    if not path.exists():
        continue
    print(f"Diagnosticando calidad de {src_id}...")
    try:
        if str(path).endswith('.csv'):
            df = pd.read_csv(path)
            table_id = path.stem
            table_issues = detect_quality_issues(df, src_id, table_id)
            all_issues.extend(table_issues)
            
            add_duplicate_quality_issues(df, src_id, table_id, all_issues)
        else:
            xl = pd.ExcelFile(path, engine='openpyxl')
            for sheet in xl.sheet_names:
                try:
                    df = pd.read_excel(path, sheet_name=sheet, engine='openpyxl')
                    is_dict = any(k in str(c).lower() for c in df.columns for k in ['variable', 'descripcion', 'tipo', 'description', 'dict'])
                    is_audit = any(k in str(c).lower() for c in df.columns for k in ['sha256', 'archivo_origen', 'fecha_extraccion', 'audit'])
                    if is_dict or is_audit or len(df) <= 1:
                        continue
                    
                    table_issues = detect_quality_issues(df, src_id, sheet)
                    all_issues.extend(table_issues)
                    
                    add_duplicate_quality_issues(df, src_id, sheet, all_issues)
                except Exception as e:
                    print(f"  Error en hoja {sheet}: {e}")
    except Exception as e:
        print(f"Error en {src_id}: {e}")

df_issues = pd.DataFrame(all_issues)
print()
print("="*70)
print("DIAGNOSTICO DE CALIDAD")
print("="*70)
if len(df_issues) > 0:
    print(f"Total issues detectados: {len(df_issues)}")
    print("\nPor severidad:")
    print(df_issues['severity'].value_counts().to_string())
    print("\nPor tipo:")
    print(df_issues['issue_type'].value_counts().head(20).to_string())
    print("\nPor fuente:")
    print(df_issues['source_id'].value_counts().to_string())
    output_path = OUTPUT_DIR / "data_quality_issues.csv"
    df_issues.to_csv(output_path, index=False)
    print(f"\nGuardado: {output_path}")
else:
    print("No se detectaron issues de calidad")

Diagnosticando calidad de iapp...
Diagnosticando calidad de microsoft...
Diagnosticando calidad de oxford...
Diagnosticando calidad de wb...
Diagnosticando calidad de wipo...
Diagnosticando calidad de stanford...
Diagnosticando calidad de oecd...
Diagnosticando calidad de anthropic...

DIAGNOSTICO DE CALIDAD
Total issues detectados: 398

Por severidad:
severity
baja     190
media    187
alta      21

Por tipo:
issue_type
constant_column          190
mostly_missing_column    147
duplicate_entity          36
empty_column              21
duplicate_row              4

Por fuente:
source_id
anthropic    159
wipo         153
oxford        71
wb             8
microsoft      3
iapp           2
oecd           2

Guardado: /home/pablo/Research_LeyIA_DataScience/Research_AI_law/outputs/eda_preliminar/data_quality_issues.csv


## BLOQUE 10 - ETAPA 9: Matriz preliminar pais x fuente

Construir vista de integrabilidad por geografia.

In [67]:
# =============================================================================
# BLOQUE 10 - ETAPA 9: Matriz preliminar pais x fuente
# =============================================================================

# Usar la cobertura ya calculada en Bloque 5
# Reconstruirla aqui para asegurar que incluye datos no nulos reales

all_countries = {}

for src_id, info in SOURCES.items():
    path = info["file"]
    if not path.exists():
        continue
    try:
        if str(path).endswith('.csv'):
            df = pd.read_csv(path)
            geo_cols = detect_geo_columns(df.columns)
            if geo_cols:
                geo_col = geo_cols[0]
                for val in df[geo_col].dropna().unique():
                    code = str(val).upper().strip()
                    if len(code) == 3 and code.isalpha():
                        name_cols = [c for c in df.columns if any(k in str(c).lower() for k in ['name', 'pais', 'country', 'economy'])]
                        subset = df[df[geo_col] == val]
                        name = subset[name_cols[0]].iloc[0] if name_cols and len(subset) > 0 else ""
                        entity_type = classify_entity(code, name)
                        if entity_type in ["country_iso3", "territory_iso3"]:
                            if code not in all_countries:
                                all_countries[code] = {"sources": set(), "names": set(), "n_attrs": 0, "entity_type": entity_type}
                            all_countries[code]["sources"].add(src_id)
                            all_countries[code]["entity_type"] = entity_type if all_countries[code].get("entity_type") != "country_iso3" else "country_iso3"
                            if pd.notna(name) and name != "":
                                all_countries[code]["names"].add(str(name))
                            if len(subset) > 0:
                                attrs = subset.notna().sum().sum()
                                all_countries[code]["n_attrs"] += int(attrs)
        else:
            xl = pd.ExcelFile(path, engine='openpyxl')
            for sheet in xl.sheet_names:
                try:
                    df = pd.read_excel(path, sheet_name=sheet, nrows=5000, engine='openpyxl')
                    is_dict = any(k in str(c).lower() for c in df.columns for k in ['variable', 'descripcion', 'tipo', 'description', 'dict'])
                    is_audit = any(k in str(c).lower() for c in df.columns for k in ['sha256', 'archivo_origen', 'fecha_extraccion', 'audit'])
                    if is_dict or is_audit or len(df) <= 1:
                        continue
                    geo_cols = detect_geo_columns(df.columns)
                    if not geo_cols:
                        continue
                    geo_col = geo_cols[0]
                    for val in df[geo_col].dropna().unique():
                        code = str(val).upper().strip()
                        if len(code) == 3 and code.isalpha():
                            name_cols = [c for c in df.columns if any(k in str(c).lower() for k in ['name', 'pais', 'country', 'economy'])]
                            subset = df[df[geo_col] == val]
                            name = subset[name_cols[0]].iloc[0] if name_cols and len(subset) > 0 else ""
                            entity_type = classify_entity(code, name)
                            if entity_type in ["country_iso3", "territory_iso3"]:
                                if code not in all_countries:
                                    all_countries[code] = {"sources": set(), "names": set(), "n_attrs": 0, "entity_type": entity_type}
                                all_countries[code]["sources"].add(src_id)
                                all_countries[code]["entity_type"] = entity_type if all_countries[code].get("entity_type") != "country_iso3" else "country_iso3"
                                if pd.notna(name) and name != "":
                                    all_countries[code]["names"].add(str(name))
                                if len(subset) > 0:
                                    attrs = subset.notna().sum().sum()
                                    all_countries[code]["n_attrs"] += int(attrs)
                except:
                    pass
    except Exception as e:
        print(f"Error en {src_id}: {e}")

matriz_rows = []
for iso3, data in sorted(all_countries.items()):
    row = {
        "iso3": iso3,
        "country_name_best_effort": " | ".join(list(data["names"])[:3]),
        "entity_type": data.get("entity_type", "country_iso3"),
    }
    for src_id in SOURCES.keys():
        row[f"present_{src_id}"] = 1 if src_id in data["sources"] else 0
    row["n_sources_present"] = len(data["sources"])
    row["n_attributes_available_total"] = data["n_attrs"]
    row["min_year_available"] = ""
    row["max_year_available"] = ""
    row["source_list"] = ", ".join(sorted(data["sources"]))
    row["integration_notes"] = ""
    matriz_rows.append(row)

df_matriz = pd.DataFrame(matriz_rows).sort_values("n_sources_present", ascending=False)

print("="*70)
print("MATRIZ PRELIMINAR PAIS X FUENTE")
print("="*70)
print(f"Total paises: {len(df_matriz)}")
print(f"Paises en 7+ fuentes: {len(df_matriz[df_matriz['n_sources_present'] >= 7])}")
print(f"Paises en 5+ fuentes: {len(df_matriz[df_matriz['n_sources_present'] >= 5])}")
print(f"Paises en 3+ fuentes: {len(df_matriz[df_matriz['n_sources_present'] >= 3])}")
print()
print("Top 15 paises con mayor integrabilidad:")
print(df_matriz.head(15)[["iso3", "country_name_best_effort", "n_sources_present", "source_list"]].to_string(index=False))
print()
print("Paises con menor cobertura (1-2 fuentes):")
print(df_matriz[df_matriz['n_sources_present'] <= 2][["iso3", "country_name_best_effort", "n_sources_present", "source_list"]].to_string(index=False))

MATRIZ PRELIMINAR PAIS X FUENTE
Total paises: 259
Paises en 7+ fuentes: 26
Paises en 5+ fuentes: 190
Paises en 3+ fuentes: 215

Top 15 paises con mayor integrabilidad:
iso3                                                    country_name_best_effort  n_sources_present                                       source_list
 TUR                                                  Turkey | Turkiye | Türkiye                  7 anthropic, iapp, oecd, oxford, stanford, wb, wipo
 BRA                                                                BRA | Brazil                  7 anthropic, iapp, oecd, oxford, stanford, wb, wipo
 GBR United Kingdom | GBR | United Kingdom of Great Britain and Northern Ireland                  7 anthropic, iapp, oecd, oxford, stanford, wb, wipo
 USA                              United States of America | USA | United States                  7 anthropic, iapp, oecd, oxford, stanford, wb, wipo
 CHN                                                                 CHN | China  

## BLOQUE 11 - ETAPA 10: Matriz preliminar pais x atributo (opcional)

In [68]:
# =============================================================================
# BLOQUE 11 - ETAPA 10: Matriz preliminar pais x atributo (AUDITORIA)
# =============================================================================
# NOTA: Esta matriz es SOLO para auditoria visual.
# NO es la matriz madre final.
# NO se imputan datos, NO se coalescen variables.

print("="*70)
print("MATRIZ PRELIMINAR PAIS X ATRIBUTO (AUDITORIA)")
print("="*70)
print()
print("Estructura conceptual:")
print("  iso3 | country_name | iapp_ley_ia_vigente | oxford_total_score_2025 | wb_gdp_per_capita_ppp_2023 | ...")
print()
print("Reglas estrictas:")
print("  - Cada variable tiene prefijo de fuente")
print("  - Se conserva el nombre original en metadata")
print("  - NO se imputan datos")
print("  - NO se crean indices compuestos")
print("  - NO se filtra por muestra final")
print()
print("Para implementacion completa, ver Fase 3 (Matriz Madre)")

MATRIZ PRELIMINAR PAIS X ATRIBUTO (AUDITORIA)

Estructura conceptual:
  iso3 | country_name | iapp_ley_ia_vigente | oxford_total_score_2025 | wb_gdp_per_capita_ppp_2023 | ...

Reglas estrictas:
  - Cada variable tiene prefijo de fuente
  - Se conserva el nombre original en metadata
  - NO se imputan datos
  - NO se crean indices compuestos
  - NO se filtra por muestra final

Para implementacion completa, ver Fase 3 (Matriz Madre)


## BLOQUE 12 - ETAPA 11: Recomendaciones para wrangling

In [69]:
# =============================================================================
# BLOQUE 12 - ETAPA 11: Recomendaciones para wrangling
# =============================================================================

recomendaciones = []
rec_id = 0

def add_rec(area, source_id, decision, default, alternatives, rationale, risk, priority):
    global rec_id
    rec_id += 1
    recomendaciones.append({
        "recommendation_id": f"REC{rec_id:03d}",
        "area": area, "source_id": source_id,
        "decision_needed": decision, "recommended_default": default,
        "alternatives": alternatives, "rationale": rationale,
        "risk_if_ignored": risk, "priority": priority,
    })

add_rec("Geografia", "all", "Definir ISO3 canonico y tabla de mapeo",
    "Usar ISO3 estandar. Mapear EU como organization_or_group. Crosswalk Microsoft.",
    "Usar nombres de pais como clave; excluir Microsoft",
    "ISO3 es el estandar internacional para merge geografico.",
    "Sin ISO3 canonico no se puede integrar geograficamente", "Alta")

add_rec("Geografia", "iapp", "Tratamiento de EU en IAPP",
    "Clasificar EU como organization_or_group, excluir de muestra de paises",
    "Incluir EU como pais; crear categoria especial",
    "EU no es un pais soberano. Distorsiona estadisticas.",
    "Sesgo de cobertura, distorsion agregada", "Alta")

add_rec("Geografia", "microsoft", "Crosswalk Microsoft -> ISO3",
    "Fuzzy matching por nombre de pais + validacion manual",
    "Excluir Microsoft de matriz madre; usar solo como contexto",
    "Microsoft no tiene ISO3. Necesita mapeo para integrarse.",
    "Sin mapeo, Microsoft queda fuera de la matriz madre", "Alta")

add_rec("Temporalidad", "oxford", "Normalizacion escalas Oxford 2019 vs 2020-2025",
    "Normalizar 2019 de 0-10 a 0-100, documentar metodologia",
    "Excluir 2019; usar solo 2020-2025; crear score separado por era",
    "2019 usa escala 0-10, 2020+ usa 0-100. Incomparables sin normalizar.",
    "Comparaciones temporales invalidas sin normalizar", "Alta")

add_rec("Variable", "oxford", "Tratamiento de 3 regimenes metodologicos",
    "Usar total_score como variable principal; documentar cambios de metodologia",
    "Crear 3 variables separadas; usar solo 2025",
    "Los 3 regimenes miden cosas ligeramente diferentes.",
    "Interpretaciones espurias al mezclar regimenes", "Alta")

add_rec("Variable", "wipo", "Usar SCORE o RANK de WIPO GII",
    "Usar SCORE (mayor = mejor). Excluir RANK o invertirlo",
    "Usar RANK (menor = mejor); usar ambos",
    "SCORE tiene direccion intuitiva. RANK requiere inversion.",
    "Interpretaciones al reves usando RANK sin invertir", "Media")

add_rec("Temporalidad", "wb", "Estrategia temporal para World Bank",
    "Snapshot: ultimo ano disponible por pais. Documentar ano usado",
    "Promedio 3 anos; usar solo 2023; panel completo",
    "WB es wide por indicador-ano. Snapshot simplifica matriz madre.",
    "Promedio sin justificar suaviza variaciones importantes", "Media")

add_rec("Variable", "stanford", "Seleccion de figuras relevantes",
    "Seleccionar figuras de Economy y Policy & Governance chapters",
    "Incluir todas las figuras; excluir Stanford",
    "Stanford tiene 341 figuras. No todas son relevantes para regulacion.",
    "Incluir figuras irrelevantes aumenta ruido", "Media")

add_rec("Fuente", "all", "Que fuentes entran a la matriz madre",
    "Todas las fuentes con ISO3 valido",
    "Solo fuentes con alta cobertura; excluir IAPP",
    "Todas las fuentes aportan informacion util.",
    "Excluir fuentes reduce potencia explicativa", "Alta")

add_rec("Variable", "iapp", "Tratamiento de texto cualitativo",
    "Separar variables textuales de variables analiticas",
    "Incluir texto como variable; hacer NLP; excluir texto",
    "Texto de IAPP es evidencia cualitativa, no variable cuantitativa.",
    "Mezclar texto con numericas complica modelamiento", "Media")

add_rec("Geografia", "all", "Definir muestra analitica final",
    "Paises con datos en >= 5 fuentes y variables regulatoria + outcome + controles",
    "Todos los paises; muestra de 86; paises comparables con Chile",
    "La muestra debe balancear cobertura y comparabilidad.",
    "Muestra muy pequena: bajo poder. Muy grande: muchos faltantes.", "Alta")

df_rec = pd.DataFrame(recomendaciones)

print("="*70)
print("RECOMENDACIONES PARA WRANGLING")
print("="*70)
print(df_rec[["recommendation_id", "area", "source_id", "priority", "decision_needed"]].to_string(index=False))
print()
for idx, row in df_rec.iterrows():
    print(f"\n{row['recommendation_id']} [{row['priority']}] {row['decision_needed']}")
    print(f"  Default: {row['recommended_default']}")
    print(f"  Riesgo: {row['risk_if_ignored']}")

output_path = OUTPUT_DIR / "recomendaciones_wrangling.csv"
df_rec.to_csv(output_path, index=False)
print(f"\nGuardado: {output_path}")

RECOMENDACIONES PARA WRANGLING
recommendation_id         area source_id priority                                decision_needed
           REC001    Geografia       all     Alta         Definir ISO3 canonico y tabla de mapeo
           REC002    Geografia      iapp     Alta                      Tratamiento de EU en IAPP
           REC003    Geografia microsoft     Alta                    Crosswalk Microsoft -> ISO3
           REC004 Temporalidad    oxford     Alta Normalizacion escalas Oxford 2019 vs 2020-2025
           REC005     Variable    oxford     Alta       Tratamiento de 3 regimenes metodologicos
           REC006     Variable      wipo    Media                  Usar SCORE o RANK de WIPO GII
           REC007 Temporalidad        wb    Media            Estrategia temporal para World Bank
           REC008     Variable  stanford    Media                Seleccion de figuras relevantes
           REC009       Fuente       all     Alta           Que fuentes entran a la matriz madre

## BLOQUE 13 - ETAPA 12: Resumen ejecutivo

In [70]:
# =============================================================================
# BLOQUE 13 - ETAPA 12: Resumen ejecutivo
# =============================================================================

df_fuentes_res = pd.read_csv(OUTPUT_DIR / "inventario_fuentes.csv")
df_tablas_res = pd.read_csv(OUTPUT_DIR / "inventario_tablas.csv")
df_variables_res = pd.read_csv(OUTPUT_DIR / "inventario_variables.csv")
df_cobertura_res = pd.read_csv(OUTPUT_DIR / "cobertura_pais_fuente.csv")
df_temporal_res = pd.read_csv(OUTPUT_DIR / "cobertura_temporal_fuente.csv")
df_issues_res = pd.read_csv(OUTPUT_DIR / "data_quality_issues.csv")
df_entidades_res = pd.read_csv(OUTPUT_DIR / "entidades_no_pais_o_sin_iso3.csv")
df_snapshot_res = pd.read_csv(OUTPUT_DIR / "variables_candidatas_snapshot.csv")
df_rec_res = pd.read_csv(OUTPUT_DIR / "recomendaciones_wrangling.csv")

present_cols = [c for c in df_cobertura_res.columns if c.startswith("present_")]
coverage_by_source = (
    df_cobertura_res[present_cols].sum().rename(lambda x: x.replace("present_", "")).sort_values(ascending=False)
)
top_sources = coverage_by_source.head(3)
low_sources = coverage_by_source.tail(3).sort_values()
issue_by_type = df_issues_res["issue_type"].value_counts().head(8)
issue_by_source = df_issues_res["source_id"].value_counts().head(8)
entity_by_type = df_entidades_res["entity_type_inferred"].value_counts().head(8)
temporal_by_structure = df_temporal_res["temporal_structure"].value_counts()
available_sources = int((df_fuentes_res["status"] == "available").sum()) if "status" in df_fuentes_res else len(df_fuentes_res)

resumen_lines = [
    "# Resumen Ejecutivo - EDA Preliminar RAW",
    "",
    "## 1. Fuentes auditadas",
    "",
    f"Se auditaron {available_sources} fuentes unificadas disponibles, {len(df_tablas_res)} tablas/hojas y {len(df_variables_res)} variables originales.",
]
for _, row in df_fuentes_res.sort_values("source_id").iterrows():
    resumen_lines.append(f"- {row['source_id']}: {row['source_name']} ({row['format']}, {int(row['n_tables_detected'])} tablas, estado={row['status']})")

resumen_lines += [
    "",
    "## 2. Cobertura geografica total",
    "",
    f"- Entidades con codigo ISO3/codigo de 3 letras detectado en al menos una fuente: {len(df_cobertura_res)}",
    f"- Entidades presentes en 5+ fuentes: {int((df_cobertura_res['n_sources_present'] >= 5).sum())}",
    f"- Entidades presentes en 3+ fuentes: {int((df_cobertura_res['n_sources_present'] >= 3).sum())}",
    f"- Entidades no pais o sin ISO3 registradas para revision: {len(df_entidades_res)}",
    "",
    "## 3. Fuentes con mayor cobertura geografica detectada",
    "",
]
for source_id, n in top_sources.items():
    resumen_lines.append(f"- {source_id}: {int(n)} entidades con codigo de 3 letras detectado")

resumen_lines += ["", "## 4. Fuentes con menor cobertura geografica detectada", ""]
for source_id, n in low_sources.items():
    resumen_lines.append(f"- {source_id}: {int(n)} entidades con codigo de 3 letras detectado")

resumen_lines += ["", "## 5. Estructura temporal", ""]
for structure, n in temporal_by_structure.items():
    resumen_lines.append(f"- {structure}: {int(n)} tablas")
resumen_lines.append(f"- Tablas candidatas a snapshot futuro: {len(df_snapshot_res)}")

resumen_lines += ["", "## 6. Principales problemas de calidad detectados", ""]
for issue_type, n in issue_by_type.items():
    resumen_lines.append(f"- {issue_type}: {int(n)} registros")
resumen_lines += ["", "Problemas por fuente:"]
for source_id, n in issue_by_source.items():
    resumen_lines.append(f"- {source_id}: {int(n)} issues")

resumen_lines += ["", "## 7. Entidades no pais o sin ISO3", ""]
for entity_type, n in entity_by_type.items():
    resumen_lines.append(f"- {entity_type}: {int(n)} registros unicos")

resumen_lines += ["", "## 8. Recomendaciones para data wrangling", ""]
for _, row in df_rec_res.sort_values("recommendation_id").iterrows():
    resumen_lines.append(f"- {row['recommendation_id']} [{row['priority']}] {row['decision_needed']}: {row['recommended_default']}")

resumen_lines += [
    "",
    "## 9. Decisiones que requieren criterio metodologico",
    "",
    "- Cual sera el outcome principal (Y)",
    "- Cual sera el tratamiento regulatorio principal (X1)",
    "- Que controles entran al modelo (X2)",
    "- Estrategia temporal: snapshot vs panel",
    "- Muestra analitica final: todos vs paises con alta cobertura",
    "- Si imputar datos faltantes",
    "",
    "---",
    "",
    "**Nota**: Este resumen fue generado desde los CSV de auditoria del notebook. Es descriptivo y neutral; no afirma si la regulacion se asocia positiva o negativamente con el ecosistema IA.",
]

resumen_text = "\n".join(resumen_lines)

output_path = OUTPUT_DIR / "resumen_eda_preliminar.md"
with open(output_path, 'w') as f:
    f.write(resumen_text)

print("="*70)
print("RESUMEN EJECUTIVO GENERADO")
print("="*70)
print(f"Guardado: {output_path}")
print()
print("Archivos generados en outputs/eda_preliminar/:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  - {f.name}")
for f in sorted(OUTPUT_DIR.glob("*.md")):
    print(f"  - {f.name}")
print()
print("EDA PRELIMINAR RAW COMPLETO")

RESUMEN EJECUTIVO GENERADO
Guardado: /home/pablo/Research_LeyIA_DataScience/Research_AI_law/outputs/eda_preliminar/resumen_eda_preliminar.md

Archivos generados en outputs/eda_preliminar/:
  - cobertura_pais_atributo.csv
  - cobertura_pais_fuente.csv
  - cobertura_temporal_fuente.csv
  - data_quality_issues.csv
  - entidades_no_pais_o_sin_iso3.csv
  - inventario_fuentes.csv
  - inventario_tablas.csv
  - inventario_variables.csv
  - recomendaciones_wrangling.csv
  - variables_candidatas_snapshot.csv
  - resumen_eda_preliminar.md

EDA PRELIMINAR RAW COMPLETO
